In [0]:
import os
import re
import json
import cv2
import pytesseract
import pandas as pd 
import logging
from PIL import Image 

In [0]:
# =============================================================================
# Logging Configuration & Dedicated Logger Manager Class
# =============================================================================
class LoggerManager:
    """Dedicated logger manager for configuring and returning the application logger."""
    @staticmethod
    def configure_logger(debug: bool = False, log_file: str = "logs/application.log") -> logging.Logger:
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        logger = logging.getLogger("PipelineLogger")

        # Set the overall logger level to DEBUG if detailed logging is desired.
        logger.setLevel(logging.DEBUG if debug else logging.INFO)

        # Remove any existing handlers. If you loop over logger.handlers directly while modifying it inside the loop, you can get errors or skip items. 

        for handler in logger.handlers[:]:    #logger.handlers[:] makes a shallow copy of the list. # Copying it first avoids that.
            logger.removeHandler(handler)
        formatter = logging.Formatter("%(asctime)s %(levelname)-8s %(message)s")

        # File Handler: records every detail.
        fh = logging.FileHandler(log_file)
        fh.setLevel(logging.DEBUG)
        fh.setFormatter(formatter)
        logger.addHandler(fh)

        # Stream Handler: shows only main logs. Creates a handler that writes log messages to a “stream” (by default sys.stderr, i.e., your console or notebook output).
        ch = logging.StreamHandler()
        ch.setLevel(logging.INFO)
        ch.setFormatter(formatter)
        logger.addHandler(ch)

        logger.debug("LoggerManager: Logger configured. Debug mode is %s.", debug)
        return logger

# Global logger instance; set debug to True to enable detailed logging.
logger = LoggerManager.configure_logger(debug=False)


In [0]:
# =============================================================================
# Shared Utility Classes
# =============================================================================
class FileUtils:
    """Utilities for handling file paths."""
    @staticmethod
    def dbfs_to_local_path(dbfs_path: str) -> str:
        logger.debug("FileUtils: Converting DBFS path '%s' to local path.", dbfs_path)
        if dbfs_path.startswith("dbfs:/"):
            local_path = os.path.join("/dbfs", dbfs_path[len("dbfs:/"):].lstrip("/"))
            logger.debug("FileUtils: Converted path is '%s'.", local_path)
            return local_path
        logger.debug("FileUtils: Path did not require conversion: '%s'.", dbfs_path)
        return dbfs_path

    @staticmethod
    def sanitize_section_name(section: str) -> str:
        sanitized = section.lower().replace(" ", "_").replace("/", "_").replace(":", "")
        logger.debug("FileUtils: Sanitized section name from '%s' to '%s'.", section, sanitized)
        return sanitized


In [0]:
class ImageUtils:
    """Image loading, preprocessing and OCR helper methods."""
    @staticmethod
    def safe_read_image(image_path: str):
        logger.debug("ImageUtils: Starting safe_read_image with image_path '%s'.", image_path)
        local_path = FileUtils.dbfs_to_local_path(image_path)
        logger.info("Reading image from: %s", local_path)

        if not os.path.exists(local_path):
            logger.error("ImageUtils: File not found: %s", local_path)
            raise FileNotFoundError(f"File not found: {local_path}")
        img = cv2.imread(local_path)

        if img is None:
            logger.error("ImageUtils: Failed to read image at: %s", local_path)
            raise ValueError(f"Failed to read image at: {local_path}")
        logger.debug("ImageUtils: Image read successfully with shape %s.", img.shape)
        return img

    @staticmethod
    # def safe_read_image_pil(image_path: str) -> Image.Image:
    #     logger.debug("ImageUtils: Starting safe_read_image_pil with image_path '%s'.", image_path)
    #     local_path = FileUtils.dbfs_to_local_path(image_path)
    #     logger.info("Reading image with PIL from: %s", local_path)

    #     if not os.path.exists(local_path):
    #         logger.error("ImageUtils: File not found: %s", local_path)
    #         raise FileNotFoundError(f"File not found: {local_path}")

    #     image = Image.open(local_path)
    #     logger.debug("ImageUtils: PIL image opened successfully.")
    #     return image

    @staticmethod
    def preprocess_image(img, debug: bool = False):
        logger.debug("ImageUtils: Starting image preprocessing. Image shape: %s", img.shape)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 15, 9
        )
        logger.debug("ImageUtils: Image preprocessing complete.")
        # if debug:
        #     import matplotlib.pyplot as plt
        #     plt.figure(figsize=(8, 8))
        #     plt.imshow(thresh, cmap="gray")
        #     plt.title("Thresholded Image")
        #     plt.axis("off")
        #     plt.show()
        return thresh

    @staticmethod
    def perform_ocr(image, config="--psm 6") -> str:
        logger.debug("ImageUtils: Performing OCR with config '%s'.", config)
        text = pytesseract.image_to_string(image, config=config)
        print(text)
        logger.debug("ImageUtils: OCR complete. Extracted text length: %d", len(text))
        return text.strip()
    
    @staticmethod
    def detect_text_regions(thresh_img, debug: bool = False):
        logger.debug("ImageUtils: Detecting text regions.")
        contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        rois = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            if w > 30 and h > 15:
                rois.append((x, y, w, h))
        rois.sort(key=lambda b: (b[1], b[0]))
        logger.debug("ImageUtils: Detected %d text regions.", len(rois))
        if debug and rois:
            img_copy = cv2.cvtColor(thresh_img, cv2.COLOR_GRAY2BGR)
            for (x, y, w, h) in rois:
                cv2.rectangle(img_copy, (x, y), (x+w, y+h), (0, 255, 0), 2)
            # import matplotlib.pyplot as plt
            # plt.figure(figsize=(10, 10))
            # plt.imshow(img_copy)
            # plt.title("Detected Text Regions")
            # plt.axis("off")
            # plt.show()
        return rois

    @staticmethod
    def perform_ocr_on_rois(img, rois, debug: bool = False):
        logger.debug("ImageUtils: Performing OCR on %d ROIs.", len(rois))
        results = []
        for (x, y, w, h) in rois:
            roi = img[y:y+h, x:x+w]
            text = pytesseract.image_to_string(roi, config="--psm 6").strip() or "[BLANK]"
            print(text)
            results.append((x, y, w, h, text))
            if debug:
                logger.debug("OCR result for ROI (%d, %d, %d, %d): %s", x, y, w, h, text)
        logger.debug("ImageUtils: Completed OCR on ROIs.")
        return results



In [0]:
# =============================================================================
# OCR Grouping Function
# =============================================================================
def group_ocr_rows(roi_results, y_threshold=20):
    logger.debug("Grouping OCR rows with y_threshold %d.", y_threshold)
    rois_with_center = [(x, y, w, h, text, y + h/2) for (x, y, w, h, text) in roi_results]
    rois_with_center.sort(key=lambda r: r[5]) # Sort each tuple by the value at index 5, which is the center. Regions are ordered from top (smallest center) to bottom (largest center) on the page.
    groups = []          # Hold all the row groups.
    current_group = []   # Collect ROIs that belong to the same text row.
    current_center = None  # Starts as None, will track the average center Y of the current_group.

    for (x, y, w, h, text, center) in rois_with_center:
        if current_center is None or abs(center - current_center) <= y_threshold: #how far vertically this new box’s center is from the running average center of the group. Absolute value, so direction (above/below) doesn’t matter.
            #If that difference is at most y_threshold pixels, it’s close enough to belong to the same “line” of text.
            current_group.append((x, y, w, h, text)) 
            current_center = center if current_center is None else (current_center + center) / 2 #If it was None, we set it to this first region’s center. Otherwise, we take the average of the existing current_center and the new region’s center. the group’s center “smoothed” as more regions join, ensuring the grouping threshold remains centered
            
        else:
            groups.append(current_group)
            current_group = [(x, y, w, h, text)]   #Starts a brand‑new group containing only this region.
            current_center = center  #Resets the running center to this region’s center for the new group.
    if current_group:
        groups.append(current_group)
    logger.debug("Grouping complete: formed %d groups.", len(groups))
    return groups

In [0]:
# =============================================================================
# Pipeline Classes
# =============================================================================

class DailyDrillingReportPipeline:
    """Processes the Daily Drilling Report section."""
    @staticmethod #  you don’t need to create an instance of the class—there’s no “state” to hold.
    def process(image_path: str, debug: bool = False):
        logger.info("DailyDrillingReportPipeline: Processing started for image '%s'.", image_path)
        logger.debug("DailyDrillingReportPipeline: Reading image.")
        img = ImageUtils.safe_read_image(image_path)
        # Crop region of interest.
        x, y, w, h = 1600, 0, 950, 185
        logger.debug("DailyDrillingReportPipeline: Cropping image region: x=%d, y=%d, w=%d, h=%d.", x, y, w, h)
        cropped = img[y:y+h, x:x+w]
        gray = cropped if len(cropped.shape) == 2 else cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
        equalized = cv2.equalizeHist(gray)  # Improves contrast—dark text vs. light background.
        blurred = cv2.GaussianBlur(equalized, (5, 5), 0) # Reduce noise and soften/smoothes out tiny speckles in the image by applying a Gaussian function.
        processed = cv2.adaptiveThreshold(blurred, 255,
                                          cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                          cv2.THRESH_BINARY, 11, 2)
        ocr_text = pytesseract.image_to_string(processed, config="--psm 6").strip()
        print(ocr_text)
        logger.info("DailyDrillingReportPipeline: OCR extraction complete.")
        logger.debug("DailyDrillingReportPipeline: OCR text: %s", ocr_text)

        # Extract keys using regex.
        expected_keys = ["Report Date", "Report Num", "Rig"]
        combined = " ".join(ocr_text.splitlines()).strip()
        combined = re.sub(r'\s+', ' ', combined)
        extracted = {}
        for i, key in enumerate(expected_keys):
            if i < len(expected_keys) - 1:
                pattern = rf'{re.escape(key)}\s*:\s*(.*?)(?=\s*{re.escape(expected_keys[i+1])}\s*:|$)' 
            else:
                pattern = rf'{re.escape(key)}\s*:\s*(.*)'  
            match = re.search(pattern, combined, re.IGNORECASE)
            extracted[key] = match.group(1).strip() if (match and match.group(1).strip()) else None
            logger.debug("DailyDrillingReportPipeline: Extracted key '%s' with value '%s'.", key, extracted[key])

        df = pd.DataFrame(list(extracted.items()), columns=["Key", "Value"])
        logger.info("DailyDrillingReportPipeline: Processing complete.")
        return {"DAILY DRILLING REPORT": extracted}, df

In [0]:

class WellJobInfoPipeline:
    """Processes the Well/Job information section."""
    @staticmethod
    def process(image_path: str, debug: bool = False):
        logger.info("WellJobInfoPipeline: Processing started for image '%s'.", image_path)
        img = ImageUtils.safe_read_image(image_path)
        ocr_text = pytesseract.image_to_string(img, config="--psm 6").strip()
        print(ocr_text)
        logger.debug("WellJobInfoPipeline: OCR text obtained with length %d.", len(ocr_text))
        expected_keys = [
            "Well Name", "Job Name", "Supervisor(s)", "Field", "Sec/Twn/Rng",
            "Phone", "AFE #", "API #", "Email", "Contractor", "Elevation",
            "RKB", "Spud Date", "Days from Spud", "Days on Loc", "MD/TVD",
            "24 Hr Footage", "Present Operations", "Activity Planned"
        ]
        combined = " ".join(ocr_text.splitlines()).strip()
        combined = re.sub(r'\s+', ' ', combined)
        result = {}
        for i, key in enumerate(expected_keys):
            if i < len(expected_keys) - 1:
                pattern = rf'{re.escape(key)}\s*:\s*(.*?)(?=\s*{re.escape(expected_keys[i+1])}\s*:|$)'
            else:
                pattern = rf'{re.escape(key)}\s*:\s*(.*)'
            match = re.search(pattern, combined, re.IGNORECASE)
            result[key] = match.group(1).strip() if match else ""
            logger.debug("WellJobInfoPipeline: Extracted key '%s' with value '%s'.", key, result[key])
        df = pd.DataFrame(list(result.items()), columns=["Key", "Value"])
        logger.info("WellJobInfoPipeline: Processing complete.")
        return {"WELL/JOB INFORMATION": result}, df


In [0]:
class MudPipeline:
    """Processes the Mud section."""
    @staticmethod
    def process(image_path: str, debug: bool = False):
        logger.info("MudPipeline: Processing started for image '%s'.", image_path)
        img = ImageUtils.safe_read_image(image_path)
        thresh_img = ImageUtils.preprocess_image(img, debug=debug)
        rois = ImageUtils.detect_text_regions(thresh_img, debug=debug)
        roi_texts = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        logger.debug("MudPipeline: Number of OCR results: %d.", len(roi_texts))
        # Build mud data using tokens.
        expected_headers = [
            "Type", "Weight In", "Weight Out", "pH", "CAKE",
            "GELS (10s/10m/30m)", "Oil/Water", "FV", "ES", "PV",
            "YP", "CL", "Ca", "LGS", "WL", "HTHP Loss", "3 RPM",
            "6 RPM", "Mud Pits and Hole Volume", "24 Hr Loss",
            "Total Loss", "Comments"
        ]
        mud_dict = MudPipeline.build_mud_dict_from_rois(roi_texts, expected_headers)
        if isinstance(mud_dict, dict):
            df = pd.DataFrame(list(mud_dict.items()), columns=["Key", "Value"])
        else:
            df = pd.DataFrame(mud_dict)
        logger.info("MudPipeline: Processing complete.")
        return {"MUD": mud_dict}, df

    @staticmethod
    def build_mud_dict_from_rois(roi_texts, expected_headers):
        logger.debug("MudPipeline: Building mud dict from %d OCR ROI results.", len(roi_texts))
        row_tolerance = 10
        rows = []
        current_row = []
        prev_y = None
        for (x, y, w, h, text) in roi_texts:
            if prev_y is None or abs(y - prev_y) <= row_tolerance:
                current_row.append((x, y, w, h, text))
            else:
                rows.append(current_row)
                current_row = [(x, y, w, h, text)]
            prev_y = y
        if current_row:
            rows.append(current_row)
        logger.debug("MudPipeline: Grouped into %d rows.", len(rows))

        row_strings = [" ".join([cell[4] for cell in sorted(row, key=lambda cell: cell[0])])
                       for row in rows]

        # Identify header and data rows.
        header1_line = None
        value1_line = None
        header2_line = None
        value2_line = None
        for i, r_text in enumerate(row_strings):
            if "type" in r_text.lower() and not header1_line:
                header1_line = r_text
                if i+1 < len(row_strings):
                    value1_line = row_strings[i+1]
            elif header1_line and not header2_line and any(kw in r_text.lower() for kw in ["rpm", "mud", "loss", "comments"]):
                header2_line = r_text
                if i+1 < len(row_strings):
                    value2_line = row_strings[i+1]
                break
        logger.debug("MudPipeline: Header1_line: %s; Header2_line: %s.", header1_line, header2_line)
        if value1_line is None:
            logger.error("MudPipeline: No data row found for Mud section!")
            return {}

        tokens1 = value1_line.split()  # Tokenize value row, pad/cut tokens to match number of headers.
        tokens2 = value2_line.split() if value2_line else []
        combined_tokens = tokens1 + tokens2
        logger.debug("MudPipeline: Combined tokens count: %d.", len(combined_tokens))
        return MudPipeline.parse_value_row_tokens(expected_headers, combined_tokens)

    @staticmethod
    def parse_value_row_tokens(expected_headers, tokens):
        logger.debug("MudPipeline: Parsing value row tokens. Expected headers: %s", expected_headers)
        expected_token_count = (len(expected_headers) - 1) + 3
        logger.debug("MudPipeline: Expected token count: %d. Actual tokens: %d.", expected_token_count, len(tokens))
        if len(tokens) < expected_token_count:
            tokens += [""] * (expected_token_count - len(tokens))
        elif len(tokens) > expected_token_count:
            tokens = tokens[:expected_token_count]
        result = {}
        idx = 0
        for header in expected_headers:
            if header == "GELS (10s/10m/30m)": # Tokenize “GELS (10s/10m/30m)” into three sub‑fields.
                gels_tokens = tokens[idx:idx+3]
                result[header] = {"10s": gels_tokens[0], "10m": gels_tokens[1], "30m": gels_tokens[2]}
                logger.debug("MudPipeline: Parsed header '%s' with tokens %s.", header, gels_tokens)
                idx += 3
            else:
                result[header] = tokens[idx]
                logger.debug("MudPipeline: Parsed header '%s' with token '%s'.", header, tokens[idx])
                idx += 1
        return result # Return a dict mapping each header to its corresponding extracted token(s).

In [0]:

# =============================================================================
# SurveyDataPipeline definition
# =============================================================================

class SurveyDataPipeline:
    
    @staticmethod
    def perform_ocr_on_rois(img, rois, debug=False):
        results = []
        n = len(rois)
        if debug and n > 0:
            cols = 5
            rows = math.ceil(n / cols)
            # fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
            # axes = axes.flatten() if rows > 1 else [axes]
        for i, (x, y, w, h) in enumerate(rois):
            roi = img[y:y+h, x:x+w]
            text = pytesseract.image_to_string(roi, config="--psm 6").strip()
            if not text:
                text = "[BLANK]"
            results.append((x, y, w, h, text))
            # if debug and i < len(axes):
            #     roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
            #     axes[i].imshow(roi_rgb)
            #     axes[i].set_title(f"ROI {i+1}\n{text[:30]}...")
            #     axes[i].axis("off")
        # if debug and n > 0:
        #     for j in range(i + 1, len(axes)):
        #         axes[j].axis("off")
        #     plt.tight_layout()
        #     plt.show()
        return results

    @staticmethod
    def build_survey_dict_from_rois(roi_texts, expected_headers, debug=False):
        row_tolerance = 10
        rows = []
        current_row = []
        prev_y = None

        # Group by similar y-coordinate. Compare Y‑coordinates to assemble boxes into lines.
        for (x, y, w, h, text) in roi_texts:
            if prev_y is None or abs(y - prev_y) <= row_tolerance:
                current_row.append((x, y, w, h, text))
            else:
                rows.append(current_row)
                current_row = [(x, y, w, h, text)]
            prev_y = y
        if current_row:
            rows.append(current_row)

        # Join texts in each row (sorted by x) and log.
        row_strings = []
        for i, row in enumerate(rows):
            row.sort(key=lambda c: c[0])
            line = " ".join(cell[4] for cell in row)
            row_strings.append(line)
            logger.info(f"Grouped Row {i}: {line}")

        # Split joined rows by newline if present.
        all_lines = []
        for line in row_strings:
            split_lines = line.split("\n")
            for subline in split_lines:
                subline = subline.strip()
                if subline:
                    all_lines.append(subline)
        logger.info(f"All extracted lines: {all_lines}")

        # Filter out header rows and incomplete rows.
        data_lines = []
        for line in all_lines:
            tokens = re.split(r'\s{2,}', line)
            if len(tokens) == 1:
                tokens = line.split()
            lower_tokens = [t.lower() for t in tokens]
            if tokens and tokens[0].upper() in {"SURVEY", "MD", "SURVEYDATA"}:
                continue
            if len(tokens) < len(expected_headers):
                logger.warning(f"Line has fewer tokens than expected: {tokens}")
                continue
            tokens = tokens[:len(expected_headers)]
            data_lines.append(tokens)

        logger.info(f"Data lines to parse: {data_lines}")

        survey_list = []
        for tokens in data_lines:
            row_dict = {expected_headers[i]: tokens[i] for i in range(len(expected_headers))}
            survey_list.append(row_dict)
        return survey_list

    @staticmethod
    def sort_survey_data(survey_list):
        def md_value(row):
            try:
                return float(row["MD"].replace(",", ""))
            except Exception:
                return 0
        return sorted(survey_list, key=md_value, reverse=True)

    @staticmethod
    def process(image_path: str, debug: bool = False):
        """
        Main processing pipeline for SURVEY DATA.
        """
        try:
            img = ImageUtils.safe_read_image(image_path)
        except Exception as e:
            logger.error(e)
            return None, None

        thresh_img = ImageUtils.preprocess_image(img, debug=debug)
        rois = ImageUtils.detect_text_regions(thresh_img, debug=debug)
        roi_texts = SurveyDataPipeline.perform_ocr_on_rois(img, rois, debug=debug)
        print("OCR Results:", roi_texts)
        expected_headers = ["MD", "Inclination", "Azimuth", "DLS", "TVD"]
        survey_list = SurveyDataPipeline.build_survey_dict_from_rois(roi_texts, expected_headers, debug=debug)
        survey_list = SurveyDataPipeline.sort_survey_data(survey_list)
        final_output = {"SURVEY DATA": survey_list}

        df = pd.DataFrame(survey_list)
        return final_output, df

In [0]:
class CostDataPipeline:
    """Processes the Cost section."""
    @staticmethod
    def process(image_path: str, debug: bool = False):
        logger.info("CostDataPipeline: Processing started for image '%s'.", image_path)
        img = ImageUtils.safe_read_image(image_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        ocr_text = pytesseract.image_to_string(gray, config="--psm 6")
        print(ocr_text)
        logger.debug("CostDataPipeline: OCR extraction complete with text length %d.", len(ocr_text))
        expected_keys = [
            "Drilling AFE Amount", "Daily Drilling Cost", "Cumulative Drilling Cost",
            "Cumulative Well Cost", "Daily Mud Cost", "Cumulative Mud Cost"
        ]
        combined = " ".join(ocr_text.splitlines()).strip()
        combined = re.sub(r'\s+', ' ', combined)
        extracted = {}
        for i, key in enumerate(expected_keys):
            if i < len(expected_keys) - 1:
                pattern = rf'{re.escape(key)}\s*:\s*(.*?)(?=\s*{re.escape(expected_keys[i+1])}\s*:|$)'
            else:
                pattern = rf'{re.escape(key)}\s*:\s*(.*)'
            match = re.search(pattern, combined, re.IGNORECASE)
            extracted[key] = match.group(1).strip() if (match and match.group(1).strip()) else "[Blank]"
            logger.debug("CostDataPipeline: Extracted key '%s' with value '%s'.", key, extracted[key])
        df = pd.DataFrame(list(extracted.items()), columns=["Key", "Value"])
        logger.info("CostDataPipeline: Processing complete.")
        return {"COST DATA": extracted}, df


In [0]:
class ObsIntPipeline:
    """Processes the Observation & Intervention section."""
    @staticmethod
    def build_obs_int_data(roi_texts):
        logger.debug("ObsIntPipeline: Building Observation & Intervention data from OCR texts.")
        
        # Define the expected order of labels.
        expected_types = ["Stop Cards", "Hazard ID's", "JSA's", "Permit to Work", "Totals"]
        # We ignore these tokens (note: we do not ignore "[BLANK]" as it is needed)
        # ignore_tokens = {"daily numbers: observation & intervention", "number"}
        
        # Flatten all OCR lines into one ordered list (assuming the ROIs are already in reading order)
        all_tokens = []
        for (_, _, _, _, text) in roi_texts:
            for line in re.split(r'\n+', text):
                token = line.strip()
                if token and token.lower() not in ignore_tokens:
                    all_tokens.append(token)
        
        logger.debug("ObsIntPipeline: All tokens after filtering headers: %s", all_tokens)
        
        # Find the first occurrence of any expected label. This anchors the main data.
        first_label_index = None
        for idx, token in enumerate(all_tokens):
            if any(token.lower() == etype.lower() for etype in expected_types):
                first_label_index = idx
                break
        if first_label_index is None:
            first_label_index = 0
        
        # Collect numeric tokens from AFTER the first expected label in the list.
        # We consider tokens that are purely numeric (or with a decimal) or exactly "[BLANK]".
        found_numbers = []
        for idx, token in enumerate(all_tokens):
            if idx > first_label_index:
                if re.match(r'^\d+(\.\d+)?$', token) or token.lower() == "[blank]":
                    # Convert "[BLANK]" to an empty string to represent a missing number.
                    found_numbers.append("" if token.lower() == "[blank]" else token)
        
        logger.debug("ObsIntPipeline: Found numeric tokens (post first label): %s", found_numbers)
        
        # Ensure we have exactly one numeric token for each expected label.
        if len(found_numbers) < len(expected_types):
            found_numbers.extend([""] * (len(expected_types) - len(found_numbers)))
        else:
            found_numbers = found_numbers[:len(expected_types)]
        
        # Zip the expected labels with the numeric tokens.
        records = [{"Type": etype, "Number": num} for etype, num in zip(expected_types, found_numbers)]
        logger.debug("ObsIntPipeline: Final structured records: %s", records)
        
        df = pd.DataFrame(records)
        logger.info("ObsIntPipeline: Observation & Intervention data processed.")
        return records, df

    @staticmethod
    def process(image_path: str, debug: bool = False):
        logger.info("ObsIntPipeline: Processing started for image '%s'.", image_path)
        img = ImageUtils.safe_read_image(image_path)
        thresh = ImageUtils.preprocess_image(img, debug=debug)
        rois = ImageUtils.detect_text_regions(thresh, debug=debug)
        roi_texts = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        print(roi_texts)
        records, df = ObsIntPipeline.build_obs_int_data(roi_texts)
        logger.info("ObsIntPipeline: Processing complete.")
        return {"DAILY NUMBERS: OBSERVATION & INTERVENTION": records}, df


In [0]:
class PersonnelPipeline:
    @staticmethod
    def preprocess_personnel_data(groups):
        """
        Extract personnel records from OCR-derived text groups.
        This function uses a regex to split the row into a header portion (for Company and Contractor)
        and the three trailing numeric values (No. Personnel, Daily Hours, Cumulative Hours).
        The header portion is then heuristically split:
            - If exactly two tokens are found, they map directly (first = Company, second = Contractor).
            - If more than two tokens are found, the last two tokens become Contractor,
              and all preceding tokens become Company.
        Totals rows (rows containing the word "totals") are handled separately.
        """
        personnel_data = []
        # List of header rows to ignore:
        # header_ignore = {
        #     "personnel", 
        #     "company contractor no. personnel daily hours cumulative hours", 
        #     "ssn"
        # }
        
        # Regex pattern to capture a row:
        # Group 1: Everything up to the three trailing numbers
        # Group 2: No. Personnel, Group 3: Daily Hours, Group 4: Cumulative Hours.
        pattern = re.compile(
            r'^(.*?)\s+(\d+(?:[.,]\d+)?)\s+(\d+(?:[.,]\d+)?)\s+(\d+(?:[.,]\d+)?)[\s]*$'
        )
        
        for group in groups:
            row_text = group.strip()
            
            # Skip header rows.
            low = row_text.lower()
            # if low in header_ignore:
            #     continue
            
            # Handle totals row separately.
            if "totals" in low:
                numeric_tokens = re.findall(r'\d+(?:[.,]\d+)?', row_text)
                if len(numeric_tokens) >= 2:
                    record = {
                        "Company": "",
                        "Contractor": "Totals",
                        "No. Personnel": "",
                        "Daily Hours": numeric_tokens[-2],
                        "Cumulative Hours": numeric_tokens[-1]
                    }
                    personnel_data.append(record)
                continue
            
            # Attempt to match the row with the regex.
            match = pattern.search(row_text)
            if not match:
                continue
            
            header_part = match.group(1).strip()
            no_personnel = match.group(2)
            daily_hours = match.group(3)
            cumulative_hours = match.group(4)
            
            header_tokens = header_part.split()
            
            if len(header_tokens) == 0:
                continue
            elif len(header_tokens) == 1:
                company = header_tokens[0]
                contractor = ""
            elif len(header_tokens) == 2:
                company = header_tokens[0]
                contractor = header_tokens[1]
            else:
                # For more than 2 tokens, assume the last two tokens are the Contractor,
                # and everything before forms the Company.
                company = " ".join(header_tokens[:-2])
                contractor = " ".join(header_tokens[-2:])
            
            record = {
                "Company": company,
                "Contractor": contractor,
                "No. Personnel": no_personnel,
                "Daily Hours": daily_hours,
                "Cumulative Hours": cumulative_hours
            }
            personnel_data.append(record)
        
        logging.info("PersonnelPipeline: Extracted %d personnel records from OCR.", len(personnel_data))
        return {"PERSONNEL": personnel_data}

    @staticmethod
    def process(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
        """
        Reads the image from the given path, performs OCR,
        groups the extracted text, and then extracts personnel records.
        """
        logging.info("PersonnelPipeline: Processing started for image '%s'.", image_path)
        
        # Read the image. (Assumes ImageUtils.safe_read_image is implemented.)
        img = ImageUtils.safe_read_image(image_path)
        
        # Convert image to grayscale and apply thresholding.
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 11, 2
        )
        
        # Detect text regions and perform OCR (Assumes these functions are defined elsewhere).
        rois = ImageUtils.detect_text_regions(thresh, debug=debug)
        roi_results = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        
        # Group OCR results by vertical position.
        groups = [
            " ".join([text for (_, _, _, _, text) in group])
            for group in group_ocr_rows(roi_results, y_threshold=20)
        ]
        
        # Process the groups to extract personnel information.
        data_dict = PersonnelPipeline.preprocess_personnel_data(groups)
        
        # Create a DataFrame if data is available.
        if data_dict.get("PERSONNEL"):
            df = pd.DataFrame(data_dict["PERSONNEL"])
        else:
            df = pd.DataFrame(columns=[
                "Company", "Contractor", "No. Personnel", "Daily Hours", "Cumulative Hours"
            ])
        
        logging.info("PersonnelPipeline: Processing complete.")
        return data_dict, df


In [0]:
class DrillBitsPipeline:
    """Processes the Drill Bits section."""
    @staticmethod
    def build_drill_bits_info(roi_texts, debug: bool = False):
        row_tolerance = 10
        grouped_rows = []
        current_row = []
        prev_y = None
        for (x, y, w, h, text) in roi_texts:
            if prev_y is None or abs(y - prev_y) <= row_tolerance:
                current_row.append((x, y, w, h, text))
            else:
                grouped_rows.append(current_row)
                current_row = [(x, y, w, h, text)]
            prev_y = y
        if current_row:
            grouped_rows.append(current_row)
        row_strings = []
        for i, row in enumerate(grouped_rows):
            row.sort(key=lambda cell: cell[0])
            line = " ".join(cell[4] for cell in row).replace("\n", " ").strip()
            row_strings.append(line)
            if debug:
                logger.info(f"Drill Bits Row {i}: {line}")
        if len(row_strings) < 3:
            logger.warning("Not enough rows for Drill Bits layout.")
            return []
        data_lines = row_strings[3:]
        final_columns = [
            "Bit #", "Size", "Make", "Model", "Serial #",
            "Nozzle-(Number x Size)", "Nozzle-TFA",
            "Depth-In", "Depth-Out", "Depth-Feet", "Depth-ROP",
            "Hours-Total", "Hours-On Btm",
            "Dull Grade-I", "Dull Grade-O1", "Dull Grade-D", "Dull Grade-L", 
            "Dull Grade-B", "Dull Grade-G", "Dull Grade-O2", "Dull Grade-RP"
        ]
        structured_data = []
        for line in data_lines:
            tokens = line.split()
            if len(tokens) < len(final_columns):
                tokens += [""] * (len(final_columns) - len(tokens))
            elif len(tokens) > len(final_columns):
                tokens = tokens[:len(final_columns)]
            row_dict = {final_columns[i]: tokens[i] for i in range(len(final_columns))}
            structured_data.append(row_dict)
            if debug:
                logger.info(f"Drill Bits Parsed row: {row_dict}")
        return structured_data

    @staticmethod
    def process(image_path: str, debug: bool = False):
        img = ImageUtils.safe_read_image(image_path)
        thresh = ImageUtils.preprocess_image(img, debug=debug)
        rois = ImageUtils.detect_text_regions(thresh, debug=debug)
        roi_texts = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        print(roi_texts)
        drill_bits = DrillBitsPipeline.build_drill_bits_info(roi_texts, debug=debug)
        logger.info("Drill Bits processed.")
        # Return None for the dataframe if not needed.
        return {"DRILL BITS": drill_bits}, None

In [0]:
class BHAPipeline:
    """Processes the BHA section."""
    @staticmethod
    def extract_bha_data(image_path: str):
        logger.debug("BHAPipeline: Extracting BHA data from image '%s'.", image_path)
        image = ImageUtils.safe_read_image(image_path) #safe_read_image_pil
        ocr_text = pytesseract.image_to_string(image)
        print(ocr_text)
        logger.debug("BHAPipeline: OCR text length: %d.", len(ocr_text))
        patterns = {
            "Drill Pipe Detail": r"Drill Pipe Detail:\s*([^\n]+)",
            "Size": r"Size:\s*([\d.]+)\b",
            "Wt./Ft": r"Wt\./Ft:\s*([\d.]+)\b",
            "Connection": r"Connection:\s*([\w\d-]+)\b",
            "ID": r"ID:\s*([\d.]+)\b",
            "Drill Bit": r"Drill Bit:\s*([^\n;]+)",
            "Motor": r"Motor:\s*([^\n;]+)",
            "MWD Tool": r"MWD Tool:\s*([^\n;]+)",
            "Monel Collar": r"Monel Collar:\s*([^\n;]+)",
            "X-Over": r"X-Over:\s*([^\n;]+)",
            "Sub": r"Sub:\s*([^\n;]+)",
            "HWDP": r"HWDP:\s*([^\n;]+)",
            "Drill Pipe": r"Drill Pipe:\s*([\d.]+(?:\" DP)?)",
            "Reamer": r"Reamer:\s*([^\n;]+)",
            "Shock Sub": r"Shock Sub:\s*([^\n;]+)",
            "Total Length": r"Total Length:\s*(\d+)\b"
        }
        bha_data = {}
        for key, pat in patterns.items():
            match = re.search(pat, ocr_text)
            if match:
                bha_data[key] = match.group(1).strip()
                logger.debug("BHAPipeline: Extracted '%s' with value '%s'.", key, bha_data[key])
        # Cleanup Drill Pipe Detail if needed.
        if "Drill Pipe Detail" in bha_data:
            detail = bha_data["Drill Pipe Detail"]
            for rem in ["Size", "Wt./Ft", "Connection", "ID"]:
                if rem in bha_data:
                    detail = re.sub(rf"{rem}:\s*{re.escape(bha_data[rem])}", "", detail).strip(",; ")
            bha_data["Drill Pipe Detail"] = detail
            logger.debug("BHAPipeline: Cleaned Drill Pipe Detail: '%s'.", detail)
        structured = {
            "BHA": {
                "Drill Pipe Detail": bha_data.get("Drill Pipe Detail", ""),
                "Size": bha_data.get("Size", ""),
                "Wt./Ft": bha_data.get("Wt./Ft", ""),
                "Connection": bha_data.get("Connection", ""),
                "ID": bha_data.get("ID", ""),
                "BHA #4": {
                    "Drill Bit": bha_data.get("Drill Bit", ""),
                    "Motor": bha_data.get("Motor", ""),
                    "MWD Tool": bha_data.get("MWD Tool", ""),
                    "Monel Collar": bha_data.get("Monel Collar", ""),
                    "X-Over": bha_data.get("X-Over", ""),
                    "Sub": bha_data.get("Sub", ""),
                    "HWDP": bha_data.get("HWDP", ""),
                    "Drill Pipe": bha_data.get("Drill Pipe", ""),
                    "Reamer": bha_data.get("Reamer", ""),
                    "Shock Sub": bha_data.get("Shock Sub", "")
                },
                "Total Length": bha_data.get("Total Length", "")
            }
        }
        logger.info("BHAPipeline: BHA data extraction complete.")
        return structured

    @staticmethod
    def process(image_path: str, debug: bool = False):
        logger.info("BHAPipeline: Processing started for image '%s'.", image_path)
        bha_json = BHAPipeline.extract_bha_data(image_path)
        df = pd.json_normalize(bha_json["BHA"])
        logger.info("BHAPipeline: Processing complete.")
        return {"BHA": bha_json["BHA"]}, df

In [0]:
class DirInfoPipeline:
    """Processes the Direction Information section."""
    @staticmethod
    def process(image_path: str, debug: bool = False):
        logger.info("DirInfoPipeline: Processing started for image '%s'.", image_path)
        img = ImageUtils.safe_read_image(image_path)
        thresh = ImageUtils.preprocess_image(img, debug=debug)
        rois = ImageUtils.detect_text_regions(thresh, debug=debug)
        roi_texts = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        print(roi_texts)
        data, df = DirInfoPipeline.build_dir_info(roi_texts)
        logger.info("DirInfoPipeline: Processing complete.")
        return data, df

    @staticmethod
    def build_dir_info(roi_texts, debug: bool = False):
        logger.debug("DirInfoPipeline: Building direction info from OCR texts.")
        all_texts = [t[4] for t in roi_texts]
        daily_cum_idx = next((i for i, txt in enumerate(all_texts)
                              if "daily" in txt.lower() and "cumulative" in txt.lower()), None)
        if daily_cum_idx is None:
            logger.warning("DirInfoPipeline: Could not locate 'Daily Cumulative' box.")
            return {}, pd.DataFrame()
        cat_idx = daily_cum_idx + 1
        if cat_idx >= len(all_texts):
            logger.warning("DirInfoPipeline: No bounding box after 'Daily Cumulative'.")
            return {}, pd.DataFrame()
        categories_box = all_texts[cat_idx]
        lines = [ln.strip() for ln in categories_box.split("\n") if ln.strip()]
        if len(lines) < 5:
            logger.warning("DirInfoPipeline: Expected at least 5 category lines, got %d.", len(lines))
        def safe_get(idx):
            return all_texts[idx] if 0 <= idx < len(all_texts) else ""
        structured = []
        for i in range(4):
            cat_name = lines[i] if i < len(lines) else f"Unknown Category {i+1}"
            daily_box = safe_get(cat_idx + 1 + (i * 2))
            cum_box = safe_get(cat_idx + 2 + (i * 2))
            structured.append({
                "Category": cat_name,
                "Daily": daily_box,
                "Cumulative": cum_box
            })
            logger.debug("DirInfoPipeline: Processed category '%s'.", cat_name)
        last_box = safe_get(cat_idx + 9)
        last_cat = lines[4] if len(lines) >= 5 else "Rotating Footage"
        tokens = last_box.replace(last_cat, "").split() if last_box else []
        daily_val = tokens[0] if len(tokens) >= 2 else ""
        cum_val = tokens[1] if len(tokens) >= 2 else ""
        structured.append({
            "Category": last_cat,
            "Daily": daily_val,
            "Cumulative": cum_val if cum_val != "]" else ""
        })
        logger.debug("DirInfoPipeline: Processed last category '%s'.", last_cat)
        df = pd.DataFrame(structured)
        return {"DIR INFO": structured}, df


In [0]:

class CasingPipeline:
    """Processes the Casing section."""
    @staticmethod
    def build_casing_dict_from_rois(roi_texts, expected_headers, debug=False):
        logger.debug("CasingPipeline: Building casing dict from OCR results with expected headers: %s", expected_headers)
        grouped_rows = group_ocr_rows(roi_texts, y_threshold=20)
        casing_rows = []
        for group in grouped_rows:
            row_string = " ".join([text for (x, y, w, h, text) in sorted(group, key=lambda item: item[0])]).strip()
            if "type" in row_string.lower() and "size" in row_string.lower():
                logger.debug("CasingPipeline: Skipping header row: %s", row_string)
                continue
            tokens = re.split(r'\s{2,}', row_string)
            if len(tokens) == 1:
                tokens = row_string.split()
            if len(tokens) < len(expected_headers):
                continue
            tokens = tokens[:len(expected_headers)]
            row_dict = {expected_headers[i]: tokens[i] for i in range(len(expected_headers))}
            casing_rows.append(row_dict)
            logger.debug("CasingPipeline: Processed row: %s", row_dict)
        logger.info("CasingPipeline: Processed %d casing rows.", len(casing_rows))
        return casing_rows

    @staticmethod
    def process(image_path: str, debug: bool = False):
        logger.info("CasingPipeline: Processing started for image '%s'.", image_path)
        img = ImageUtils.safe_read_image(image_path)
        thresh = ImageUtils.preprocess_image(img, debug=debug)
        rois = ImageUtils.detect_text_regions(thresh, debug=debug)
        roi_texts = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        print(roi_texts)
        expected_headers = ["Type", "Size", "Weight", "Grade", "Connection", "Top MD", "Bottom MD", "TOC"]
        casing_data = CasingPipeline.build_casing_dict_from_rois(roi_texts, expected_headers, debug=debug)
        df = pd.DataFrame(casing_data)
        logger.info("CasingPipeline: Processing complete.")
        return {"CASING": casing_data}, df


In [0]:
class ConsumablesPipeline:
    """Processes the Consumables section."""
    @staticmethod
    def group_rois_by_row(roi_results, threshold: int = 20):
        logger.debug("ConsumablesPipeline: Grouping ROIs by row with threshold %d.", threshold)
        roi_with_center = [(x, y, w, h, text, y + h/2) for (x, y, w, h, text) in roi_results]
        roi_with_center.sort(key=lambda r: r[5])
        groups = []
        current_group = []
        current_center = None
        for (x, y, w, h, text, y_center) in roi_with_center:
            if current_center is None or abs(y_center - current_center) < threshold:
                current_group.append(text)
                current_center = y_center if current_center is None else (current_center + y_center) / 2
            else:
                groups.append(" ".join(current_group))
                current_group = [text]
                current_center = y_center
        if current_group:
            groups.append(" ".join(current_group))
        logger.debug("ConsumablesPipeline: Formed %d groups.", len(groups))
        return groups

    @staticmethod
    def build_consumables_dict_from_rois(roi_texts, debug: bool = False):
        logger.debug("ConsumablesPipeline: Building consumables dict from OCR results.")
        groups = ConsumablesPipeline.group_rois_by_row(roi_texts, threshold=20)
        data_rows = []
        for line in groups:
            line_str = line.strip()
            # if ("consumable" in line_str.lower() and "received" in line_str.lower()) or "nun" in line_str.lower():
            #     continue
            if len(line_str.split()) < 5:
                continue
            data_rows.append(line_str)
        consumables_list = []
        for line in data_rows:
            tokens = re.split(r'\s+', line)
            if len(tokens) > 5:
                first = " ".join(tokens[:-4])
                tokens = [first] + tokens[-4:]
            if len(tokens) != 5:
                continue
            record = {
                "Consumable": tokens[0],
                "Daily Received (gal)": tokens[1],
                "Daily Used (gal)": tokens[2],
                "Cumulative Used (gal)": tokens[3],
                "Daily on Hand (gal)": tokens[4]
            }
            consumables_list.append(record)
            logger.debug("ConsumablesPipeline: Processed record: %s", record)
        logger.info("ConsumablesPipeline: Completed processing consumables.")
        return consumables_list

    @staticmethod
    def process(image_path: str, debug: bool = False):
        logger.info("ConsumablesPipeline: Processing started for image '%s'.", image_path)
        img = ImageUtils.safe_read_image(image_path)
        thresh = ImageUtils.preprocess_image(img, debug=debug)
        rois = ImageUtils.detect_text_regions(thresh, debug=debug)
        roi_texts = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        print(roi_texts)
        consumables_list = ConsumablesPipeline.build_consumables_dict_from_rois(roi_texts, debug=debug)
        df = pd.DataFrame(consumables_list)
        logger.info("ConsumablesPipeline: Processing complete.")
        return {"CONSUMABLES": consumables_list}, df


In [0]:

# =============================================================================
# Pumps Pipeline: Handles Pumps Table and Drilling/Circ Rates Parsing
# =============================================================================
class PumpsPipeline:
    @staticmethod
    def perform_ocr(img: Image.Image) -> str:
        logger.info("PumpsPipeline: Performing OCR using PIL.")
        text = pytesseract.image_to_string(img)
        print(text)
        logger.info("PumpsPipeline: OCR extraction complete. Text length: %d", len(text))
        return text

    @staticmethod
    def parse_pumps_table(ocr_text: str) -> list:
        logger.debug("PumpsPipeline: Parsing pumps table from OCR text.")
        # pump_pattern = re.compile(
        #     r"^(\d+)\s+(BOMCO)\s+(TRIPLEX)\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)$",
        #     re.IGNORECASE
        # )
        pump_pattern = re.compile(
            r"^(\d+)\s+([^\s]+)\s+([^\s]+)\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)$",
            re.IGNORECASE
        )

        pumps = []
        for line in ocr_text.splitlines():
            line = line.strip()
            print(line)
            match = pump_pattern.match(line)
            if match:
                (number, model, pump_type, hhp, efficiency, stroke, liner,
                 p_rating, p_limit, spm_rating, spm_limit) = match.groups()
                pump_data = {
                    "Number": number,
                    "Model": model.upper(),
                    "Type": pump_type.upper(),
                    "HHP": hhp,
                    "Efficiency": efficiency,
                    "Stroke (in)": stroke,
                    "Liner (in)": liner,
                    "P-Rating (psi)": p_rating,
                    "P-Limit (psi)": p_limit,
                    "SPM Rating": spm_rating,
                    "SPM Limit": spm_limit
                }
                pumps.append(pump_data)
                logger.debug("PumpsPipeline: Parsed pump row: %s", pump_data)
        logger.info("PumpsPipeline: Extracted %d pump rows.", len(pumps))
        return pumps

    @staticmethod
    def parse_drilling_circ_rates(ocr_text: str) -> list:
        logger.debug("PumpsPipeline: Parsing drilling/circ rate section from OCR text.")
        tokens = [t.strip() for t in ocr_text.splitlines() if t.strip()]
        print(tokens)
        start_idx = None
        for i, token in enumerate(tokens):
            if token.lower().startswith("drilling") and "rate" in token.lower():
                start_idx = i
                logger.debug("PumpsPipeline: Found drilling section starting at index %d.", i)
                break
        if start_idx is None:
            logger.warning("PumpsPipeline: No drilling section found in OCR text.")
            return []
        drill_tokens = tokens[start_idx:]
        # Group tokens into blocks of 10
        rows = []
        for i in range(0, len(drill_tokens), 10):
            group = drill_tokens[i:i+10]
            if len(group) < 10:
                break
            rows.append(group)
        parsed_rows = []
        for group in rows:
            rate_match = re.search(r"(\d+)", group[0])
            rate_id = rate_match.group(1) if rate_match else ""
            pressure_match = re.search(r"([\d\.]+)", group[1])
            pressure = pressure_match.group(1) if pressure_match else ""
            spm = group[3] if group[3].isdigit() else ""
            gal_match = re.search(r"([\d\.]+)", group[5])
            gal_stroke = gal_match.group(1) if gal_match else ""
            gpm_match = re.search(r"([\d\.]+)", group[6])
            gpm = gpm_match.group(1) if gpm_match else ""
            bpm_match = re.search(r"([\d\.]+)", group[7])
            bpm = bpm_match.group(1) if bpm_match else ""
            dc_match = re.search(r"([\d\.]+)", group[8])
            dc = dc_match.group(1) if dc_match else ""
            dp_match = re.search(r"([\d\.]+)", group[9])
            dp = dp_match.group(1) if dp_match else ""
            parsed = {
                "RateID": rate_id,
                "Pressure": pressure,
                "SPM": spm,
                "Gal_Stroke": gal_stroke,
                "GPM": gpm,
                "BPM": bpm,
                "DC": dc,
                "DP": dp
            }
            parsed_rows.append(parsed)
            logger.debug("PumpsPipeline: Parsed drilling row: %s", parsed)
        logger.info("PumpsPipeline: Extracted %d drilling/circ rate rows.", len(parsed_rows))
        return parsed_rows

    @staticmethod
    def process(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
        logger.info("PumpsPipeline: Processing image '%s'.", image_path)
        img = ImageUtils.safe_read_image(image_path)
        ocr_text = PumpsPipeline.perform_ocr(img)
        print(ocr_text)
        pumps = PumpsPipeline.parse_pumps_table(ocr_text)
        drilling = PumpsPipeline.parse_drilling_circ_rates(ocr_text)
        final_data = {
            "PUMPS": {
                "PUMPS": pumps,
                "DrillingCircRates": drilling
            }
        }
        df = pd.DataFrame(pumps)
        logger.info("PumpsPipeline: Processing complete.")
        return final_data, df


In [0]:
# =============================================================================
# BOP-Specific Utilities & Functions with Detailed Logging
# =============================================================================

def safe_read_image_bop(image_path: str):
    """
    Top-level function to read an image using OpenCV for the BOP section.
    """
    local_path = FileUtils.dbfs_to_local_path(image_path)
    logger.info("BOP: Reading image from: %s", local_path)
    if not os.path.exists(local_path):
        logger.error("BOP: File not found: %s", local_path)
        raise FileNotFoundError(f"File not found: {local_path}")
    img = cv2.imread(local_path)
    if img is None:
        logger.error("BOP: OpenCV failed to load image: %s", local_path)
        raise ValueError(f"OpenCV failed to load image: {local_path}")
    logger.debug("BOP: Image read successfully with shape %s", img.shape)
    return img

def safe_read_image_pil_bop(image_path: str) -> Image.Image:
    local_path = FileUtils.dbfs_to_local_path(image_path)
    logger.info("BOP: Reading image (PIL) from: %s", local_path)
    if not os.path.exists(local_path):
        logger.error("BOP: File not found: %s", local_path)
        raise FileNotFoundError(f"File not found: {local_path}")
    image = Image.open(local_path)
    logger.debug("BOP: PIL image opened successfully.")
    return image

def perform_ocr_bop(img, config="--psm 6") -> str:
    logger.debug("BOP: Converting image to grayscale for OCR.")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    text = pytesseract.image_to_string(gray, config=config)
    logger.debug("BOP: OCR extraction complete with text length %d.", len(text))
    return text.strip()

def group_ocr_rows_bop(roi_results, y_threshold=20):
    """
    Group OCR result bounding boxes by their y-coordinate.
    """
    logger.debug("BOP: Grouping OCR rows with threshold %d.", y_threshold)
    rois_with_center = [(x, y, w, h, text, y + h/2) for (x, y, w, h, text) in roi_results]
    rois_with_center.sort(key=lambda r: r[5])
    groups = []
    current_group = []
    current_center = None
    for (x, y, w, h, text, center) in rois_with_center:
        if current_center is None:
            current_group.append((x, y, w, h, text))
            current_center = center
        elif abs(center - current_center) <= y_threshold:
            current_group.append((x, y, w, h, text))
            current_center = (current_center + center) / 2
        else:
            groups.append(current_group)
            current_group = [(x, y, w, h, text)]
            current_center = center
    if current_group:
        groups.append(current_group)
    logger.debug("BOP: Grouping complete: %d groups formed.", len(groups))
    return groups

class ImageUtils_bop:
    """Shared image utilities for BOP processing with detailed logging."""
    @staticmethod
    def safe_read_image_bop(image_path: str):
        logger.debug("ImageUtils_bop: Invoking safe_read_image_bop.")
        return safe_read_image_bop(image_path)
    
    # @staticmethod
    # def safe_read_image_pil_bop(image_path: str) -> Image.Image:
    #     logger.debug("ImageUtils_bop: Invoking safe_read_image_pil_bop.")
    #     return safe_read_image_pil_bop(image_path)

    @staticmethod
    def preprocess_image_bop(img, debug: bool = False):
        logger.debug("ImageUtils_bop: Preprocessing image.")
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 15, 9
        )
        logger.debug("ImageUtils_bop: Image preprocessing complete.")
        return thresh

    @staticmethod
    def detect_text_regions_bop(thresh_img, debug: bool = False):
        logger.debug("ImageUtils_bop: Detecting text regions.")
        contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        rois = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            if w > 30 and h > 15:
                rois.append((x, y, w, h))
        rois.sort(key=lambda b: (b[1], b[0]))
        logger.debug("ImageUtils_bop: Found %d text regions.", len(rois))
        return rois

    @staticmethod
    def perform_ocr_on_rois_bop(img, rois, debug: bool = False): #No image‑drawing or plotting, only logs each region’s text if debug=True.
        logger.debug("ImageUtils_bop: Performing OCR on %d regions.", len(rois))
        results = []
        for (x, y, w, h) in rois:
            roi = img[y:y+h, x:x+w]
            text = pytesseract.image_to_string(roi, config="--psm 6").strip() or "[BLANK]"
            results.append((x, y, w, h, text))
            if debug:
                logger.debug("ImageUtils_bop: OCR result for region (%d, %d, %d, %d): %s", x, y, w, h, text)
        return results

# =============================================================================
# BOP Pipeline: Encapsulates OCR and Parsing for BOP Data
# =============================================================================
class BOPPipeline:
    @staticmethod
    def extract_bop_info_bop(text: str) -> dict:
        logger.debug("BOPPipeline: Extracting BOP information from OCR text.")
        patterns = {
            "Last BOP Test Date": r"Last BOP Test Date\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})",
            "Last BOP Drill": r"Last BOP Drill\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})",
            "Next BOP Test": r"Next BOP Test\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})"
        }
        result = {}
        for key, regex in patterns.items():
            match = re.search(regex, text, re.IGNORECASE)
            result[key] = match.group(1) if match else ""
            logger.debug("BOPPipeline: Extracted %s = %s", key, result[key])
        return result

    @staticmethod
    def process_bop(image_path: str, debug: bool = False) -> (dict, pd.DataFrame):
        logger.info("BOPPipeline: Processing image '%s'", image_path)
        img = safe_read_image_bop(image_path)
        ocr_text = perform_ocr_bop(img, config="--psm 6")
        print(ocr_text)
        bop_info = BOPPipeline.extract_bop_info_bop(ocr_text)
        for key, value in bop_info.items():
            if value:
                logger.info("BOPPipeline: %s -> %s", key, value)
            else:
                logger.warning("BOPPipeline: %s not found in OCR text.", key)
        df = pd.DataFrame(list(bop_info.items()), columns=["Key", "Value"])
        logger.info("BOPPipeline: Processing complete.")
        return {"BOP": bop_info}, df


In [0]:
# =============================================================================
# TimeBreakdownPipeline definition
# =============================================================================

class TimeBreakdownPipeline:

    @staticmethod
    def perform_ocr_on_rois(img, rois, debug=True):
        results = []
        n = len(rois)
        if debug and n > 0:
            cols = 5
            rows = math.ceil(n / cols)
            fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
            axes = axes.flatten() if rows > 1 else [axes]
        for i, (x, y, w, h) in enumerate(rois):
            roi = img[y:y+h, x:x+w]
            text = pytesseract.image_to_string(roi, config="--psm 6").strip()
            if not text:
                text = "[BLANK]"
            results.append((x, y, w, h, text))
            # if debug and i < len(axes):
            #     roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
                # axes[i].imshow(roi_rgb)
                # axes[i].set_title(f"ROI {i+1}\n{text[:30]}...")
                # axes[i].axis("off")
        # if debug and n > 0:
        #     for j in range(i + 1, len(axes)):
        #         axes[j].axis("off")
            # plt.tight_layout()
            # plt.show()
        return results

    @staticmethod
    def group_ocr_rows(roi_results, y_threshold=20):
        groups = []
        roi_results_sorted = sorted(roi_results, key=lambda r: r[1])
        current_group = []
        current_y = None
        for (x, y, w, h, text) in roi_results_sorted:
            if current_y is None:
                current_y = y
                current_group.append((x, y, w, h, text))
            elif abs(y - current_y) <= y_threshold:
                current_group.append((x, y, w, h, text))
            else:
                groups.append(current_group)
                current_group = [(x, y, w, h, text)]
                current_y = y
        if current_group:
            groups.append(current_group)
        return groups

    @staticmethod
    def parse_operations_description(ops_text):
        ops_data = {
            "Depth": {"From": "", "To": ""},
            "Performance": {"Feet": "", "FPH": ""},
            "Rotation_Slide": {"Rotate": "", "Slide": ""},
            "Rotation_Time": {"Rotate Time": "", "Slide Time": ""},
            "GPM": "",
            "MTR RPM": "",
            "SPP": "",
            "DIFF": "",
            "WOB": "",
            "ROT RPM": "",
            "ON BTM TRQ": "",
            "OFF BTM TRQ": "",
            "GAS": {"Units": "", "Flare": ""},
            "MW": {"In": "", "Out": ""},
            "Targets": [],
            "Observations": []
        }
        depth_match = re.search(r"F/\s*([\d,']+)\s*T/\s*([\d,']+)", ops_text, re.IGNORECASE)
        if depth_match:
            ops_data["Depth"]["From"] = depth_match.group(1)
            ops_data["Depth"]["To"] = depth_match.group(2)
        perf_match = re.search(r"\(([\d,']+)\s*@\s*(\d+)\s*FPH\)", ops_text, re.IGNORECASE)
        if perf_match:
            ops_data["Performance"]["Feet"] = perf_match.group(1)
            ops_data["Performance"]["FPH"] = perf_match.group(2)
        rs_match = re.search(r"ROTATE\s*([\d.]+%)\s*/\s*SLIDE\s*([\d.]+%)", ops_text, re.IGNORECASE)
        if rs_match:
            ops_data["Rotation_Slide"]["Rotate"] = rs_match.group(1)
            ops_data["Rotation_Slide"]["Slide"] = rs_match.group(2)
        rt_match = re.search(r"ROTATE\s*TIME\s*([\d.]+%)\s*/\s*SLIDE\s*TIME\s*([\d.]+%)", ops_text, re.IGNORECASE)
        if rt_match:
            ops_data["Rotation_Time"]["Rotate Time"] = rt_match.group(1)
            ops_data["Rotation_Time"]["Slide Time"] = rt_match.group(2)
        numeric_patterns = {
            "GPM": r"GPM:\s*(\d+)",
            "MTR RPM": r"MTR\s*RPM:\s*(\d+)",
            "SPP": r"SPP:\s*([\d,]+(?:-\d+)?)(?:,|\s|$)",
            "DIFF": r"DIFF:\s*([\d\-]+)",
            "WOB": r"WOB:\s*([\d,]+(?:-\d+)?)(?:,|\s|$)",
            "ROT RPM": r"ROT\s*RPM:\s*([\d,]+(?:-\d+)?)(?:,|\s|$)",
            "ON BTM TRQ": r"ON\s*BTM\s*TRQ[:;]?\s*([\d\-K]+)",
            "OFF BTM TRQ": r"OFF\s*BTM\s*TRQ[:;]?\s*([\d\-K]+)"
        }
        for key, pattern in numeric_patterns.items():
            m = re.search(pattern, ops_text, re.IGNORECASE)
            if m:
                ops_data[key] = m.group(1)
        gas_units = re.search(r"GAS:\s*([\d,]+)\s*UNITS", ops_text, re.IGNORECASE)
        if gas_units:
            ops_data["GAS"]["Units"] = gas_units.group(1)
        flare = re.search(r"(NO\s*FLARE|FLARE\s*ON|FLARE\s*\S+)", ops_text, re.IGNORECASE)
        if flare:
            ops_data["GAS"]["Flare"] = flare.group(1)
        mw_match = re.search(r"MW\s*IN\s*([\d.+]+)\s*PPG\s*/\s*OUT\s*([\d.+]+)\s*PPG", ops_text, re.IGNORECASE)
        if mw_match:
            ops_data["MW"]["In"] = mw_match.group(1)
            ops_data["MW"]["Out"] = mw_match.group(2)
        header_match = re.search(r".*MW\s*IN\s*[\d.+]+\s*PPG\s*/\s*OUT\s*[\d.+]+\s*PPG\.", ops_text, re.IGNORECASE)
        if header_match:
            residual = ops_text[header_match.end():]
        else:
            residual = ops_text
        segments = re.split(r'(?=\*\*\*)', residual)
        obs_list = []
        for seg in segments:
            seg = seg.strip()
            if not seg:
                continue
            if not seg.startswith('***'):
                parts = [p.strip() for p in seg.split('.') if p.strip()]
                obs_list.extend(parts)
            else:
                obs_list.append(seg)
        obs_list = [o.lstrip('* ').strip() for o in obs_list]
        clean_obs = [o for o in obs_list if "TARGET" not in o.upper()]
        targets = [o for o in obs_list if "TARGET" in o.upper()]
        targets = [t.lstrip('* ').strip() for t in targets]
        ops_data["Observations"] = clean_obs
        ops_data["Targets"] = targets
        return ops_data

    @staticmethod
    def parse_row_text(row_text):
        clean_text = " ".join(row_text.split())
        if "Daily Hrs" in clean_text:
            pattern = r"Daily Hrs\s+(\S+)\s+Daily NPT Hrs\s*(\S*)\s+Total Job NPT Hours\s+(\S+)"
            m = re.search(pattern, clean_text, re.IGNORECASE)
            if m:
                return {
                    "Daily Summary": {
                        "Daily Hrs": m.group(1),
                        "Daily NPT Hrs": m.group(2),
                        "Total Job NPT Hours": m.group(3)
                    }
                }
            else:
                logger.warning(f"Daily summary row detected but could not parse: {clean_text}")
                return None
        tokens = clean_text.split()
        if not tokens or not re.match(r"\d{2}:\d{2}", tokens[0]):
            logger.info(f"Skipping header or invalid row: {clean_text}")
            return None
        if len(tokens) < 8:
            logger.warning(f"Row does not have enough tokens: {clean_text}")
            return None
        from_time = tokens[0]
        to_time = tokens[1]
        hours = tokens[2]
        depth_start = tokens[3]
        depth_end = tokens[4]
        header_rest = " ".join(tokens[5:])
        m = re.search(r"^(?P<phase>.+?)\s+(?P<activity>DR[-]?Drilling)\s+(?P<ops>.*)$", header_rest, re.IGNORECASE)
        if m:
            phase = m.group("phase")
            activity = m.group("activity")
            ops = m.group("ops")
        else:
            phase = tokens[5]
            activity = tokens[6] if len(tokens) > 6 else ""
            ops = " ".join(tokens[7:]) if len(tokens) > 7 else ""
        return {
            "From": from_time,
            "To": to_time,
            "Hours": hours,
            "Depth Start": depth_start,
            "Depth End": depth_end,
            "Phase": phase,
            "Activity": activity,
            "Operations Description": TimeBreakdownPipeline.parse_operations_description(ops)
        }

    @staticmethod
    def parse_all_rows_from_text(full_text):
        rows = []
        daily_summary = None
        if re.search(r"\d{2}:\d{2}\s+\d{2}:\d{2}", full_text):
            row_chunks = re.split(r"(?=\d{2}:\d{2}\s+\d{2}:\d{2})", full_text)
            for chunk in row_chunks:
                chunk = chunk.strip()
                if not chunk:
                    continue
                row = TimeBreakdownPipeline.parse_row_text(chunk)
                if row:
                    if "Daily Summary" in row:
                        daily_summary = row
                    else:
                        rows.append(row)
        else:
            fallback_row = {
                "From": "",
                "To": "",
                "Hours": "",
                "Depth Start": "",
                "Depth End": "",
                "Phase": "",
                "Activity": "",
                "Operations Description": TimeBreakdownPipeline.parse_operations_description(full_text)
            }
            return [fallback_row]
        if daily_summary is not None:
            rows.append(daily_summary)
        return rows

    @staticmethod
    def parse_all_rows_from_ocr_groups(roi_results):
        rows = []
        daily_summary = None
        groups = TimeBreakdownPipeline.group_ocr_rows(roi_results, y_threshold=20)
        for group in groups:
            group_sorted = sorted(group, key=lambda r: r[0])
            row_text = " ".join([text for (x, y, w, h, text) in group_sorted])
            if any(kw in row_text.upper() for kw in ["TIME PERIOD", "FROM TO", "DEPTH PHASE", "OPERATIONS DESCRIPTION"]):
                continue
            parsed_row = TimeBreakdownPipeline.parse_row_text(row_text)
            if parsed_row:
                if "Daily Summary" in parsed_row:
                    daily_summary = parsed_row
                else:
                    rows.append(parsed_row)
        if daily_summary is not None:
            rows.append(daily_summary)
        return rows

    @staticmethod
    def merge_time_breakdown_data(main_data, continuation_data):
        return main_data + continuation_data

    @staticmethod
    def process_time_breakdown_image(img_path, debug=False):
        img = ImageUtils.safe_read_image(img_path)
        thresh_img = ImageUtils.preprocess_image(img, debug=debug)
        rois = ImageUtils.detect_text_regions(thresh_img, debug=debug)
        roi_ocr_results = ImageUtils.perform_ocr_on_rois(img, rois, debug=debug)
        rows = TimeBreakdownPipeline.parse_all_rows_from_ocr_groups(roi_ocr_results)
        if not rows:
            full_text = pytesseract.image_to_string(thresh_img, config="--psm 6")
            rows = TimeBreakdownPipeline.parse_all_rows_from_text(full_text)
        return rows

# =============================================================================
# Special handling for TIME BREAKDOWN (Merge Two Pages)
# =============================================================================

def process_time_breakdown_section(debug: bool = False) -> (dict, pd.DataFrame):
    logger.info("Processing TIME BREAKDOWN from two pages.")
    # Process two known page images
    tb1 = TimeBreakdownPipeline.process_time_breakdown_image("dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_14.png", debug=debug)
    tb2 = TimeBreakdownPipeline.process_time_breakdown_image("dbfs:/mnt/mini-proj-dd/cropped_sections/page_2_section_1.png", debug=debug)
    merged_tb = tb1 + tb2
    data = {"TIME BREAKDOWN": merged_tb}
    df = pd.json_normalize(merged_tb) if merged_tb else pd.DataFrame()
    logger.info("TIME BREAKDOWN: Merged %d rows from two pages.", len(merged_tb))
    return data, df

In [0]:
# =============================================================================
# Final Integrated Main Pipeline
# =============================================================================

def main(debug: bool = False):
    global logger
    logger = LoggerManager.configure_logger(debug=debug, log_file="logs/application.log")
    logger.info("Main Pipeline: Starting main pipeline execution...")

    # Define image paths for all sections.
    image_paths = {
        "DAILY DRILLING REPORT": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_1.png",
        "WELL/JOB INFORMATION": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_2.png",
        "MUD": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_3.png",
        "SURVEY DATA": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_4.png",
        "DIR INFO": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_5.png",
        "DRILL BITS": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_6.png",
        "CASING": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_7.png",
        "BOP": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_8.png",
        "PERSONNEL": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_9.png",
        "DAILY NUMBERS: OBSERVATION & INTERVENTION": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_10.png",
        "BHA": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_11.png",
        "PUMPS": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_12.png",
        "COST DATA": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_13.png",
        "CONSUMABLES": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_2_section_2.png",
        "BIT INFO": "dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_6.png",
        "TIME BREAKDOWN": "SPECIAL"  # Special handling below.
    }

    # Define pipeline functions for each section.
    pipelines = {
        "DAILY DRILLING REPORT": DailyDrillingReportPipeline.process,
        "WELL/JOB INFORMATION": WellJobInfoPipeline.process,
        "MUD": MudPipeline.process,
        "SURVEY DATA": SurveyDataPipeline.process,
        "DIR INFO": DirInfoPipeline.process,
        "DRILL BITS": DrillBitsPipeline.process,
        "CASING": CasingPipeline.process,
        "BOP": BOPPipeline.process_bop,
        "PERSONNEL": PersonnelPipeline.process,
        "DAILY NUMBERS: OBSERVATION & INTERVENTION": ObsIntPipeline.process,
        "BHA": BHAPipeline.process,
        "PUMPS": PumpsPipeline.process,
        "COST DATA": CostDataPipeline.process,
        "TIME BREAKDOWN": process_time_breakdown_section,
        "CONSUMABLES": ConsumablesPipeline.process,
    }

    output_folder = FileUtils.dbfs_to_local_path("dbfs:/mnt/mini-proj-dd/final_results")
    os.makedirs(output_folder, exist_ok=True)
    aggregated_json = {}
    aggregated_df = pd.DataFrame()

    # Process sections based on image_paths.
    for section, img_path in image_paths.items():
        if section == "TIME BREAKDOWN":
            func = pipelines.get("TIME BREAKDOWN")
            try:
                # For TIME BREAKDOWN, do not pass an img_path argument.
                data_json, df = func(debug=debug)
                aggregated_json[section] = data_json.get(section, data_json)
                safe_section = FileUtils.sanitize_section_name(section)
                json_file = os.path.join(output_folder, f"{safe_section}.json")
                with open(json_file, "w") as f:
                    json.dump(data_json, f, indent=4)
                if df is not None and not df.empty:
                    aggregated_df = pd.concat([aggregated_df, df], ignore_index=True)
                    csv_file = os.path.join(output_folder, f"{safe_section}.csv")
                    df.to_csv(csv_file, index=False)
                logger.info("Section '%s' processed successfully.", section)
            except Exception as e:
                logger.exception("Error processing section '%s': %s", section, e)
            continue

        pipeline_key = section
        func = pipelines.get(pipeline_key)
        if func is None:
            logger.info("Skipping section '%s' — pipeline not implemented.", section)
            continue
        try:
            logger.info("Processing section '%s' from %s...", section, img_path)
            data_json, df = func(img_path, debug=debug)
            safe_section = FileUtils.sanitize_section_name(section)
            aggregated_json[section] = data_json.get(section, data_json)
            json_file = os.path.join(output_folder, f"{safe_section}.json")
            with open(json_file, "w") as f:
                json.dump(data_json, f, indent=4)
            if df is not None and not df.empty:
                aggregated_df = pd.concat([aggregated_df, df], ignore_index=True)
                csv_file = os.path.join(output_folder, f"{safe_section}.csv")
                df.to_csv(csv_file, index=False)
            logger.info("Section '%s' processed successfully.", section)
        except Exception as e:
            logger.exception("Error processing section '%s': %s", section, e)

    # Save aggregated outputs.
    agg_json_path = os.path.join(output_folder, "aggregated_data.json")
    with open(agg_json_path, "w") as f:
        json.dump(aggregated_json, f, indent=4)
    agg_csv_path = os.path.join(output_folder, "aggregated_data.csv")
    aggregated_df.to_csv(agg_csv_path, index=False)
    logger.info("Aggregated results saved to %s and %s.", agg_json_path, agg_csv_path)
    print("----- Aggregated JSON Output -----")
    print(json.dumps(aggregated_json, indent=4))

# =============================================================================
# Entry Point
# =============================================================================

if __name__ == "__main__":
    main(debug=False)


2025-04-18 21:09:04,563 INFO     Main Pipeline: Starting main pipeline execution...
2025-04-18 21:09:05,055 INFO     Processing section 'DAILY DRILLING REPORT' from dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_1.png...
2025-04-18 21:09:05,056 INFO     DailyDrillingReportPipeline: Processing started for image 'dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_1.png'.
2025-04-18 21:09:05,057 INFO     Reading image from: /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_1.png
2025-04-18 21:09:17,261 INFO     DailyDrillingReportPipeline: OCR extraction complete.
2025-04-18 21:09:17,264 INFO     DailyDrillingReportPipeline: Processing complete.
2025-04-18 21:09:17,333 INFO     Section 'DAILY DRILLING REPORT' processed successfully.
2025-04-18 21:09:17,334 INFO     Processing section 'WELL/JOB INFORMATION' from dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_2.png...
2025-04-18 21:09:17,335 INFO     WellJobInfoPipeline: Processing started for image 'dbfs:/mnt/mini-

Report Date: 7/4/2024
Report Num: 11.
Rig: Cyclone 39


2025-04-18 21:09:21,663 INFO     WellJobInfoPipeline: Processing complete.
2025-04-18 21:09:21,708 INFO     Section 'WELL/JOB INFORMATION' processed successfully.
2025-04-18 21:09:21,709 INFO     Processing section 'MUD' from dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_3.png...
2025-04-18 21:09:21,709 INFO     MudPipeline: Processing started for image 'dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_3.png'.
2025-04-18 21:09:21,710 INFO     Reading image from: /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_3.png


WELL/JOB INFORMATION
Well Name: Ross Fee 4371-31-7-15 MH Job Name: Drilling Supervisor(s): CHAD MILLER / ED COOLEY
Field: XBE Sec/Twn/Rng: 31, 43N, 71W Phone: 307-315-1908
AFE #: 240098 API #: 49-005-78911 Email: cyclone39@aec-denver.com
Contractor: Elevation: 4913.5 RKB: 27.5
Spud Date: 6/4/2024 Days from Spud: 7.67 Days on Loc: 34
MD/TVD: 20537 FT/10719 FT 24 Hr Footage: 3068
Present Operations: DRILLING LATERAL @ 20,537'.
Activity Planned: DRILL LATERAL SECTION TO PLANNED TD @ ~21,226', PUMP TD SWEEPS & CHC, SOOH & L/D DRILL PIPE.
MUD
MUD
[BLANK]
Type Weight In Weight Out pH CAKE
GELS (10s/10m/30m)
Oil/Water FV ES PV YP CL Ca LGS WL HTHP Loss
OBM
11.5
11.5
[BLANK]
3
8
25
27
88/12
60.0
753
16
8
31,000
326,667
4.47
[BLANK]
5.00
3 RPM 6 RPM Mud Pits and Hole Volume 24 Hr Loss Total Loss
Comments
4
5
1023
13


2025-04-18 21:09:27,746 INFO     MudPipeline: Processing complete.
2025-04-18 21:09:27,787 INFO     Section 'MUD' processed successfully.
2025-04-18 21:09:27,788 INFO     Processing section 'SURVEY DATA' from dbfs:/mnt/mini-proj-dd/cropped_sections/page_1_section_4.png...
2025-04-18 21:09:27,789 INFO     Reading image from: /dbfs/mnt/mini-proj-dd/cropped_sections/page_1_section_4.png


...

DRILL LATERAL SECTION F/ 19,660' T/ 20,252' (592' @ 148 FPH) (ROTATE 93.7% / SLIDE 6.3%) (ROTATE
TIME 73.1% / SLIDE TIME 26.9%). GPM: 350, MTR RPM: 263, SPP: 4,425, DIFF: 900-1,100, WOB: 30-32,
ROT RPM: 40-50, ON BTM TRQ: 9-13K, OFF BTM TRQ: 3-6K, GAS: 2,962 UNITS / NO FLARE, MW IN 11.5+
PPG / OUT 11.5 PPG.

*** MONITOR G/L - MINIMAL LOSSES @ ~1 BPH.

***HOLDING 500 PSI ON CONNECTIONS FOR WELLBORE STABILITY.

***PUMP 10-BBL (18-PPB) LCM SWEEPS TO MITIGATE MUD SEEPAGE, EVERY 300'-500'.

***TARGET #15: 10,665.0' TVD @ 0' VS W/ 89.7 INC TO 10,000' VS (10,717.4' TVD), TOLERANCE 5.0°
ABOVE AND 5.0' BELOW THE LINE, 25' RIGHT / LEFT.
04:00 04:30 0.50 20252 20252 Production Drilling - DR-Rig Service RIG SERVICE.
Lateral
***GREASE WASH PIPE & BLOCKS.
***CHECK TOP DRIVE OIL & TOP OFF.
***PERFORM STATIC BRAKE TEST.
***SPR @ 20,252', CMW @ 11.5 PPG.
***MP #2: 20 / 625 PSI, 30 / 755 PSI, 40 / 868 PSI.
04:30 06:00 1.50 20252 20537 Production Drilling - DR-Drilling DRILL LATERAL SECTION F/ 20,252' T/

2025-04-18 21:10:42,230 INFO     Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
INFO:PipelineLogger:Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
2025-04-18 21:10:42,232 INFO     Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
INFO:PipelineLogger:Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
2025-04-18 21:10:42,233 INFO     Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
INFO:PipelineLogger:Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
2025-04-18 21:10:42,234 INFO     Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
INFO:PipelineLogger:Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLA

[BLANK]
Daily Hrs 24.00 Daily NPT Hrs Total Job NPT Hours 5.00


INFO:PipelineLogger:Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
2025-04-18 21:10:42,244 INFO     Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
INFO:PipelineLogger:Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
2025-04-18 21:10:42,246 INFO     Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
INFO:PipelineLogger:Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
2025-04-18 21:10:42,247 INFO     Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
INFO:PipelineLogger:Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK] [BLANK]
2025-04-18 21:10:42,247 INFO     Skipping header or invalid row: [BLANK] [BLANK] [BLANK] [BLA

----- Aggregated JSON Output -----
{
    "DAILY DRILLING REPORT": {
        "Report Date": "7/4/2024",
        "Report Num": "11.",
        "Rig": "Cyclone 39"
    },
    "WELL/JOB INFORMATION": {
        "Well Name": "Ross Fee 4371-31-7-15 MH",
        "Job Name": "Drilling",
        "Supervisor(s)": "CHAD MILLER / ED COOLEY",
        "Field": "XBE",
        "Sec/Twn/Rng": "31, 43N, 71W",
        "Phone": "307-315-1908",
        "AFE #": "240098",
        "API #": "49-005-78911",
        "Email": "cyclone39@aec-denver.com",
        "Contractor": "",
        "Elevation": "4913.5",
        "RKB": "27.5",
        "Spud Date": "6/4/2024",
        "Days from Spud": "7.67",
        "Days on Loc": "34",
        "MD/TVD": "20537 FT/10719 FT",
        "24 Hr Footage": "3068",
        "Present Operations": "DRILLING LATERAL @ 20,537'.",
        "Activity Planned": "DRILL LATERAL SECTION TO PLANNED TD @ ~21,226', PUMP TD SWEEPS & CHC, SOOH & L/D DRILL PIPE."
    },
    "MUD": {
        "Type": "